In [6]:
from functions_newparams import *
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import h5py
from matplotlib.backends.backend_pdf import PdfPages
%matplotlib widget

fig, (ax,ax2,ax3) = plt.subplots(1,3,figsize=(12,4))

fit_params = [0.0, 0.005, 3, [1.8, 1.8], 0.90, 1.0, np.linspace(5,18,200)]
pars = 6
cs = list(cm.Greens(np.arange(pars) / pars) ) 

count = 0
for sig, ls, fillcolor in zip([0.7,0.5,0.2,0.0],['solid','dashed','dotted', 'dashdot'],['purple', 'blue', 'teal', 'orange']):
    qlf = QLF(1, fit_params[1])
    qlf.get_dNdlnMstar(sig)
    slopes = qlf.get_slope(qlf.get_Mhalo(qlf.StellBins))

    qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
    qlf.get_dNdlnL(fit_params[-1], fit_params[3])
    
    
    halos = qlf.get_Mhalo(qlf.StellBins)

    if sig == 0:
        ax.plot(10**halos, 10**qlf.StellBins, c='k', lw = 1)
    ax.set_xlabel('$M_h(M_\odot)$',fontsize=12)
    ax.set_ylabel('$M_*(M_\odot)$',fontsize=12)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_ylim([10**7,10**12])
    ax.set_xlim([10**10, 10**15])
    if count == 0:
        for c, mass in zip(cs, [7,8,9,10,11,12,13]):
            ax.axhline(10**mass, ls = 'dotted', c=c)
            ax.axhspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)   
    if sig != 0:
        ax.fill_between(10**halos, np.e**(np.log(10**qlf.StellBins)-sig), np.e**(np.log(10**qlf.StellBins)+sig), color=fillcolor, label='$\sigma_{\lnM_*} = $'+str(sig), alpha=1)
    
#     stells = qlf.get_Mstar(qlf.HaloBins)
#     if sig == 0:
#         ax.plot(10**qlf.HaloBins, 10**stells, c='k', lw = 1)
#     ax.set_xlabel('$M_h(M_\odot)$',fontsize=12)
#     ax.set_ylabel('$M_*(M_\odot)$',fontsize=12)
#     ax.set_xscale('log')
#     ax.set_yscale('log')
#     ax.set_ylim([10**7,10**12])
#     ax.set_xlim([10**10, 10**15])
#     if count == 0:
#         for c, mass in zip(cs, [7,8,9,10,11,12,13]):
#             ax.axhline(10**mass, ls = 'dotted', c=c)
#             ax.axhspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)   
#     if sig != 0:
#         ax.fill_between(10**qlf.HaloBins, np.e**(np.log(10**stells)-sig), np.e**(np.log(10**stells)+sig), color=fillcolor, label='$\sigma_{\lnM_*} = $'+str(sig), alpha=1)
    
    
    if sig == 0:
        ax2.plot(10**qlf.StellBins, slopes, c='k', lw=1)
    ax2.set_ylabel('d$\lnM_* / $d$\lnM_h$')
    ax2.set_xlabel('$M_*(M_\odot)$',fontsize=12)
    ax2.set_xscale('log')
    ax2.set_ylim([0,2.5])
    ax2.set_xlim([10**7, 10**12])
    ax2.tick_params(axis='both', which='both', labelsize=10, direction='in')
    if count == 0:
        for c, mass in zip(cs, [7,8,9,10,11,12,13]):
            ax2.axvline(10**mass, ls = 'dotted', c=c)
            ax2.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label='$10^{'+f'{mass}'+r'}< M_* (M_\odot) < 10^{'+f'{mass+1}'+'}$')
            ax2.legend(fontsize = 8, framealpha = 1)
    if sig != 0:
        ax2.fill_betweenx(slopes, np.e**(np.log(10**qlf.StellBins)-sig), np.e**(np.log(10**qlf.StellBins)+sig), color=fillcolor, alpha = 1)



    ax3.plot(10**qlf.StellBins, np.log10(qlf.dNdlnMstar), c='k', label='$\sigma_{\lnM_*} = $'+str(sig), ls=ls, lw = 1)
    ax3.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [M_*]^{-1})$',fontsize=12)
    ax3.set_xlabel('$M_*(M_\odot)$',fontsize=12)
    ax3.set_xscale('log')
    ax3.set_xlim([10**7,10**12])
    ax3.set_ylim([-7,0])
    ax3.tick_params(axis='both', which='both', labelsize=10, direction='in')
    if count == 0:
        for c, mass in zip(cs, [7,8,9,10,11,12,13]):
            ax3.axvline(10**mass, ls = 'dotted', c=c)
            ax3.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)

    count += 1
    
ax3.legend(fontsize = 8)
ax.legend(fontsize = 8)
plt.tight_layout()

plt.savefig('plots/exploreSMFscatter.pdf', transparent = True)


FigureCanvasNbAgg()

In [38]:
'''
SINGLE QLF
'''
from functions import *
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import h5py
from matplotlib.backends.backend_pdf import PdfPages
%matplotlib widget
import glob as glob

files = np.array(glob.glob('umachine-dr1/observational_constraints/*.smf'))
zlow = np.array([float(f.split('_z')[1]) for f in files])
zhigh = np.array([float(f.split('_z')[2].split('.s')[0]) for f in files])
inds = np.where((zlow <= 1.0) & (zhigh >= 1.0))[0]
print(len(inds),'SMF files found within redshift range:')
for f in files[inds]:
    print(f)
smf_masses_tot = []
smf_tot = []
smf_errdown_tot = []
smf_errup_tot = []
for f in files[inds]:
    data = np.loadtxt(f)
    masses = (data[:,0] + data[:,1]) / 2.
    smf = data[:,2]
    smf_errdown = data[:,3]
    smf_errup = data[:,4]
    smf_masses_tot.extend(list(masses))
    smf_tot.extend(list(smf))
    smf_errdown_tot.extend(list(smf_errdown))
    smf_errup_tot.extend(list(smf_errup))
    
    
files = np.array(glob.glob('umachine-dr1/observational_constraints/*.ssfr'))
zlow = np.array([float(f.split('_z')[1]) for f in files])
zhigh = np.array([float(f.split('_z')[2].split('.s')[0]) for f in files])
inds = np.where((zlow <= 1.0) & (zhigh >= 1.0))[0]
print(len(inds),'SSFR files found within redshift range:')
for f in files[inds]:
    print(f)
ssfr_masses_tot = []
ssfr_tot = []
ssfr_errdown_tot = []
ssfr_errup_tot = []
for f in files[inds]:
    data = np.loadtxt(f)
    masses = (data[:,0] + data[:,1]) / 2.
    ssfr = data[:,2]
    ssfr_errdown = 10**ssfr - 10**(ssfr - data[:,3])
    ssfr_errup = 10**(ssfr + data[:,4]) - 10**ssfr
    ssfr_masses_tot.extend(list(masses))
    ssfr_tot.extend(list(ssfr))
    ssfr_errdown_tot.extend(list(ssfr_errdown))
    ssfr_errup_tot.extend(list(ssfr_errup))
    
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(11,4))

fit_params = [0.2, 0.005, 3, [1.8, 1.8], 0.90, 1.0, np.linspace(5,18,200)]

pars = 6
cs = list(cm.Greens(np.arange(pars) / pars) ) 

qlf = QLF(1, fit_params[1])
qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
qlf.get_dNdlnL(fit_params[-1], fit_params[3])
ax1.plot(10**qlf.StellBins, qlf.SSFRs,c='k')
ax1.errorbar(10**np.array(ssfr_masses_tot), 10**np.array(ssfr_tot), yerr = [np.array(ssfr_errdown_tot), np.array(ssfr_errup_tot)], fmt = '.', c = 'r', markersize=5)
ax1.set_ylabel('Observed SSFR (yr$^{-1}$)',fontsize=12)
ax1.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlim([10**7,10**12])
ax1.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax1.axvline(10**mass, ls = 'dotted', c=c)
    ax1.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label=f'{mass}'+r'$< \log [M_*](M_\odot) <$'+f'{mass+1}')
    ax1.legend(fontsize = 8, framealpha = 1)



ax2.plot(10**qlf.StellBins, np.log10(qlf.dNdlnMstar), c='k')
ax2.errorbar(10**np.array(smf_masses_tot), np.array(smf_tot), yerr = [np.array(smf_errdown_tot), np.array(smf_errup_tot)], fmt = '.', c = 'r', markersize=5)
ax2.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [M_*]^{-1})$',fontsize=12)
ax2.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax2.set_xscale('log')
ax2.set_xlim([10**7,10**12])
ax2.set_ylim([-8,-1])
ax2.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax2.axvline(10**mass, ls = 'dotted', c=c)
    ax2.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)

plt.tight_layout()

plt.savefig('plots/SMFvsObs.pdf', transparent = True)

print(qlf.StellBins[-1] - qlf.StellBins[-2])
lnStellBins = np.log(10**qlf.StellBins)
print(lnStellBins[-1] - lnStellBins[-2])
print(lnStellBins[1] - lnStellBins[0])
print(qlf.StellBins[1] - qlf.StellBins[0])



5 SMF files found within redshift range:
umachine-dr1/observational_constraints/moustakas_z0.80_z1.00.smf
umachine-dr1/observational_constraints/tomczak_z0.75_z1.0.smf
umachine-dr1/observational_constraints/muzzin_ilbert_z0.5_z1.1.smf
umachine-dr1/observational_constraints/muzzin_ilbert_z1.0_z1.5.smf
umachine-dr1/observational_constraints/tomczak_z1.0_z1.25.smf
11 SSFR files found within redshift range:
umachine-dr1/observational_constraints/kajisawa09_z1.0_z1.5.ssfr
umachine-dr1/observational_constraints/whitaker14_z1.0_z1.5.ssfr
umachine-dr1/observational_constraints/karim11_z0.8_z1.0.ssfr
umachine-dr1/observational_constraints/tomczak15_z1.00_z1.25.ssfr
umachine-dr1/observational_constraints/zwart14_z0.5_z1.0.ssfr
umachine-dr1/observational_constraints/zwart14_z1.0_z1.5.ssfr
umachine-dr1/observational_constraints/karim11_z1.0_z1.2.ssfr
umachine-dr1/observational_constraints/schreiber15_z0.7_z1.2.ssfr
umachine-dr1/observational_constraints/tomczak15_z0.75_z1.00.ssfr
umachine-dr1/obse

FigureCanvasNbAgg()

0.0050026329647181456
0.011518988090280402
0.01151898809028129
0.0050026329647181456


In [43]:
'''
SINGLE QLF
'''
from functions import *
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import h5py
from matplotlib.backends.backend_pdf import PdfPages
%matplotlib widget


def Shen_fit_uncer(z, lums, ver): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params

    def shen_func(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(z, [d0]) + C.chebval(1 + z, [0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_b(p):
        L = lums
        a0, a1, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = a0*zfrac**a1
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)

    params = {'a0':[0.85858, 0.03092, 0.02876], 'a1':[-0.26236, 0.02003, 0.01753], 'a2':[0.02105, 0.00136, 0.00113],\
        'b0':[2.54992, 0.01915, 0.02949], 'b1':[-1.04735, 0.01815, 0.02999], 'b2':[1.13277, 0.01988, 0.03891],\
        'c0':[13.01297, 0.00943, 0.01354], 'c1':[-0.57587, 0.00205, 0.00261], 'c2':[0.45361, 0.00290, 0.00434],\
        'd0':[-3.53138, 0.02694, 0.02690], 'd1':[-0.39961, 0.00871, 0.00896]}
    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    
    params_b = {'a0':[0.3653, 0.0115, 0.0114], 'a1':[-0.6006, 0.0422, 0.0417],\
        'b0':[2.4709,0.0163,0.0169], 'b1':[-0.9963,0.0167,0.0161], 'b2':[1.0716, 0.0180, 0.0181],\
        'c0':[12.9656,0.0092,0.0089], 'c1':[-0.5758,0.0020,0.0019], 'c2':[0.4698,0.0025,0.0026],\
        'd0':[-3.6276,0.0209, 0.0203], 'd1':[-0.3444,0.0063,0.0061]}
    
    
    if ver == 'orig':
        param_list = np.array([params[i] for i in params])
        NUM = int(1e4)
        rand_params = get_params(params)
        ys = np.apply_along_axis(shen_func, 1, rand_params).T
        ya = shen_func(param_list[:,0])
    elif ver == 'a':
        param_list = np.array([params_a[i] for i in params_a])
        NUM = int(1e4)
        rand_params = get_params(params_a)
        ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
        ya = shen_func_a(param_list[:,0]) 
    elif ver == 'b':
        param_list = np.array([params_b[i] for i in params_b])
        NUM = int(1e4)
        rand_params = get_params(params_b)
        ys = np.apply_along_axis(shen_func_b, 1, rand_params).T
        ya = shen_func_b(param_list[:,0]) 

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw





def QLFwShen_test(fit_params = None, z = 0.0, name = 'z0-QLF-v-Shen.pdf', Hopkins = False, approx_local=True):
   

    ### what fit params are we using
    if not fit_params:
        siglnM = 0.7
        bins = 0.005
        start = 10.0
        siglnX = [3.0, 2.0]
        lums = np.linspace(5,18,200)
    else:
        siglnM, bins, start, siglnX, slope, norm, lums = fit_params
    
    qlf = QLF(z, bins)
    qlf.get_Mbh(start, slope, norm, approx_local=approx_local)

    m = qlf.slopes

    qlf.get_dNdlnL(lums, siglnX)
    lumsp = 10**lums*3.8e33
    prea = np.zeros(len(lumsp))
    posta = np.zeros(len(lumsp))

    for i, pre, m in zip(np.transpose(qlf.intvals), qlf.pre, qlf.slopes):
        dens = i
        if pre == True:
            prea += dens
        else:
            posta += dens

    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF')
    
    pars = 6
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.2, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)

        mass_begin += 1

    lumsshen = lums
    xshen = 10**lumsshen*3.8e33

    dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'a')
    
    ax.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
    
#     ax.axvline(10**8.95*3.8e33,c='k',linestyle='dotted')
    ax.axvline(10**6.5*3.8e33,c='r',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
    ### formatting and save
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-3,'z = '+str(z),fontsize = 12)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    
    

f = h5py.File("output/chi2_SHEN_r100_v2.3.2_w0_s2.h5py", "r") 
siglnX2 = f['siglnX2'][:]
chi23d = f['z=1.0/chi23d_grid'][:]
f.close()
minval = np.min(chi23d)
minind = np.where(chi23d == minval)
bestpost = siglnX2[minind[0][0]]

print('\nShen best fits (linear SMBH relation): minval =',minval)
print(f'Best post-disk = {bestpost:.2f}')


fig, (ax, ax2, ax3, ax4) = plt.subplots(1,4,figsize=(15.5,3.5))

fit_params = [0.2, 0.005, 3, [1.8, bestpost], 0.90, 1.0, np.linspace(5,18,200)]

ax.text(10**48.75, -4, '$\sigma_{\ln X} = $'+str(fit_params[3][1])[0:3])
QLFwShen_test(z=1, fit_params=fit_params, approx_local = True)

pars = 6
cs = list(cm.Greens(np.arange(pars) / pars) ) 


qlf = QLF(1, fit_params[1])
qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
qlf.get_dNdlnL(fit_params[-1], fit_params[3])
ax2.plot(10**qlf.StellBins, qlf.SSFRs,c='k')

ax2.set_ylabel('Observed SSFR (yr$^{-1}$)',fontsize=12)
ax2.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim([10**7,10**12.5])
ax2.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax2.axvline(10**mass, ls = 'dotted', c=c)
    ax2.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label=f'{mass}'+r'$< \log [M_*](M_\odot) <$'+f'{mass+1}')
    ax2.legend(fontsize = 8, framealpha = 1)



ax3.plot(10**qlf.StellBins, np.log10(qlf.dNdlogMstar), c='k')
ax3.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [M_*]^{-1})$',fontsize=12)
ax3.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax3.set_xscale('log')
ax3.set_xlim([10**7,10**12.5])
ax3.set_ylim([-7,0])
ax3.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax3.axvline(10**mass, ls = 'dotted', c=c)
    ax3.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
    
ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='~Local')
ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
ax4.set_yscale('log')
ax4.set_xscale('log')
ax4.set_xlim([10**7,10**12.5])
ax4.set_ylim([10**3, 10**10])
ax4.legend(fontsize = 8, framealpha=1)
ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax4.axvline(10**mass, ls = 'dotted', c=c)
    ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)

plt.tight_layout()



plt.savefig('plots/SMFstart_bestfit-linearSMBH-extendedL.pdf', transparent = True)




Shen best fits (linear SMBH relation): minval = 27.513410472474263
Best post-disk = 2.09


FigureCanvasNbAgg()

In [45]:
'''
SINGLE QLF
'''
from functions import *
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import h5py
from matplotlib.backends.backend_pdf import PdfPages
%matplotlib widget


def Shen_fit_uncer(z, lums, ver): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params

    def shen_func(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(z, [d0]) + C.chebval(1 + z, [0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_b(p):
        L = lums
        a0, a1, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = a0*zfrac**a1
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)

    params = {'a0':[0.85858, 0.03092, 0.02876], 'a1':[-0.26236, 0.02003, 0.01753], 'a2':[0.02105, 0.00136, 0.00113],\
        'b0':[2.54992, 0.01915, 0.02949], 'b1':[-1.04735, 0.01815, 0.02999], 'b2':[1.13277, 0.01988, 0.03891],\
        'c0':[13.01297, 0.00943, 0.01354], 'c1':[-0.57587, 0.00205, 0.00261], 'c2':[0.45361, 0.00290, 0.00434],\
        'd0':[-3.53138, 0.02694, 0.02690], 'd1':[-0.39961, 0.00871, 0.00896]}
    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    
    params_b = {'a0':[0.3653, 0.0115, 0.0114], 'a1':[-0.6006, 0.0422, 0.0417],\
        'b0':[2.4709,0.0163,0.0169], 'b1':[-0.9963,0.0167,0.0161], 'b2':[1.0716, 0.0180, 0.0181],\
        'c0':[12.9656,0.0092,0.0089], 'c1':[-0.5758,0.0020,0.0019], 'c2':[0.4698,0.0025,0.0026],\
        'd0':[-3.6276,0.0209, 0.0203], 'd1':[-0.3444,0.0063,0.0061]}
    
    
    if ver == 'orig':
        param_list = np.array([params[i] for i in params])
        NUM = int(1e4)
        rand_params = get_params(params)
        ys = np.apply_along_axis(shen_func, 1, rand_params).T
        ya = shen_func(param_list[:,0])
    elif ver == 'a':
        param_list = np.array([params_a[i] for i in params_a])
        NUM = int(1e4)
        rand_params = get_params(params_a)
        ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
        ya = shen_func_a(param_list[:,0]) 
    elif ver == 'b':
        param_list = np.array([params_b[i] for i in params_b])
        NUM = int(1e4)
        rand_params = get_params(params_b)
        ys = np.apply_along_axis(shen_func_b, 1, rand_params).T
        ya = shen_func_b(param_list[:,0]) 

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw





def QLFwShen_test(fit_params = None, z = 0.0, name = 'z0-QLF-v-Shen.pdf', Hopkins = False, approx_local=True):
   

    ### what fit params are we using
    if not fit_params:
        siglnM = 0.7
        bins = 0.005
        start = 10.0
        siglnX = [3.0, 2.0]
        lums = np.linspace(5,18,200)
    else:
        siglnM, bins, start, siglnX, slope, norm, lums = fit_params
    
    qlf = QLF(z, bins)
    qlf.get_Mbh(start, slope, norm, approx_local=approx_local)

    m = qlf.slopes

    qlf.get_dNdlnL(lums, siglnX)
    lumsp = 10**lums*3.8e33
    prea = np.zeros(len(lumsp))
    posta = np.zeros(len(lumsp))

    for i, pre, m in zip(np.transpose(qlf.intvals), qlf.pre, qlf.slopes):
        dens = i
        if pre == True:
            prea += dens
        else:
            posta += dens

    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF')
    
    pars = 6
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.2, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)

        mass_begin += 1

    lumsshen = lums
    xshen = 10**lumsshen*3.8e33

    dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'a')
    
    ax.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
    
#     ax.axvline(10**8.95*3.8e33,c='k',linestyle='dotted')
    ax.axvline(10**6.5*3.8e33,c='r',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
    ### formatting and save
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-3,'z = '+str(z),fontsize = 12)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    
    

f = h5py.File("output/chi2_SHEN_r30_v2.4.2_w0_s2.h5py", "r") 
siglnX2 = f['siglnX2'][:]
norm_locals = np.linspace(6, 10, 30)
chi23d = f['z=1.0/chi23d_grid'][:].T
f.close()
minval = np.amin(chi23d)
minind = np.where(chi23d == minval)
bestpost = siglnX2[minind[1][0]]
bestnorm = norm_locals[minind[0][0]]


print('\nShen best fits (linear SMBH relation): minval =',minval)
print(f'Best post-disk = {bestpost:.2f},    Best local-norm = {bestnorm:.2f}')


fig, (ax, ax2, ax3, ax4) = plt.subplots(1,4,figsize=(15.5,3.5))

fit_params = [0.2, 0.005, 3, [1.8, bestpost], 0.90, 1.0, np.linspace(5,18,200)]

ax.text(10**48.75, -4, '$\sigma_{\ln X} = $'+str(fit_params[3][1])[0:3])
QLFwShen_test(z=1, fit_params=fit_params, approx_local = True)

pars = 6
cs = list(cm.Greens(np.arange(pars) / pars) ) 


qlf = QLF(1, fit_params[1])
qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True, norm_local = bestnorm)
qlf.get_dNdlnL(fit_params[-1], fit_params[3])
ax2.plot(10**qlf.StellBins, qlf.SSFRs,c='k')

ax2.set_ylabel('Observed SSFR (yr$^{-1}$)',fontsize=12)
ax2.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim([10**7,10**12.5])
ax2.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax2.axvline(10**mass, ls = 'dotted', c=c)
    ax2.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label=f'{mass}'+r'$< \log [M_*](M_\odot) <$'+f'{mass+1}')
    ax2.legend(fontsize = 8, framealpha = 1)



ax3.plot(10**qlf.StellBins, np.log10(qlf.dNdlogMstar), c='k')
ax3.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [M_*]^{-1})$',fontsize=12)
ax3.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax3.set_xscale('log')
ax3.set_xlim([10**7,10**12.5])
ax3.set_ylim([-7,0])
ax3.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax3.axvline(10**mass, ls = 'dotted', c=c)
    ax3.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
    
ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='~Local')
ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
ax4.set_yscale('log')
ax4.set_xscale('log')
ax4.set_xlim([10**7,10**12.5])
ax4.set_ylim([10**3, 10**10])
ax4.legend(fontsize = 8, framealpha=1)
ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax4.axvline(10**mass, ls = 'dotted', c=c)
    ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)

plt.tight_layout()



plt.savefig('plots/SMFstart_bestfit-linearSMBH-freepostNorm-extendedL.pdf', transparent = True)




Shen best fits (linear SMBH relation): minval = 15.26652992298107
Best post-disk = 2.55,    Best local-norm = 7.79


FigureCanvasNbAgg()

In [47]:
'''
SINGLE QLF
'''
from functions import *
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import h5py
from matplotlib.backends.backend_pdf import PdfPages
%matplotlib widget


def Shen_fit_uncer(z, lums, ver): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params

    def shen_func(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(z, [d0]) + C.chebval(1 + z, [0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_b(p):
        L = lums
        a0, a1, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = a0*zfrac**a1
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)

    params = {'a0':[0.85858, 0.03092, 0.02876], 'a1':[-0.26236, 0.02003, 0.01753], 'a2':[0.02105, 0.00136, 0.00113],\
        'b0':[2.54992, 0.01915, 0.02949], 'b1':[-1.04735, 0.01815, 0.02999], 'b2':[1.13277, 0.01988, 0.03891],\
        'c0':[13.01297, 0.00943, 0.01354], 'c1':[-0.57587, 0.00205, 0.00261], 'c2':[0.45361, 0.00290, 0.00434],\
        'd0':[-3.53138, 0.02694, 0.02690], 'd1':[-0.39961, 0.00871, 0.00896]}
    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    
    params_b = {'a0':[0.3653, 0.0115, 0.0114], 'a1':[-0.6006, 0.0422, 0.0417],\
        'b0':[2.4709,0.0163,0.0169], 'b1':[-0.9963,0.0167,0.0161], 'b2':[1.0716, 0.0180, 0.0181],\
        'c0':[12.9656,0.0092,0.0089], 'c1':[-0.5758,0.0020,0.0019], 'c2':[0.4698,0.0025,0.0026],\
        'd0':[-3.6276,0.0209, 0.0203], 'd1':[-0.3444,0.0063,0.0061]}
    
    
    if ver == 'orig':
        param_list = np.array([params[i] for i in params])
        NUM = int(1e4)
        rand_params = get_params(params)
        ys = np.apply_along_axis(shen_func, 1, rand_params).T
        ya = shen_func(param_list[:,0])
    elif ver == 'a':
        param_list = np.array([params_a[i] for i in params_a])
        NUM = int(1e4)
        rand_params = get_params(params_a)
        ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
        ya = shen_func_a(param_list[:,0]) 
    elif ver == 'b':
        param_list = np.array([params_b[i] for i in params_b])
        NUM = int(1e4)
        rand_params = get_params(params_b)
        ys = np.apply_along_axis(shen_func_b, 1, rand_params).T
        ya = shen_func_b(param_list[:,0]) 

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw





def QLFwShen_test(fit_params = None, z = 0.0, name = 'z0-QLF-v-Shen.pdf', Hopkins = False, approx_local=True):
   

    ### what fit params are we using
    if not fit_params:
        siglnM = 0.7
        bins = 0.005
        start = 10.0
        siglnX = [3.0, 2.0]
        lums = np.linspace(5,18,200)
    else:
        siglnM, bins, start, siglnX, slope, norm, lums = fit_params
    
    qlf = QLF(z, bins)
    qlf.get_Mbh(start, slope, norm, approx_local=approx_local)

    m = qlf.slopes

    qlf.get_dNdlnL(lums, siglnX)
    lumsp = 10**lums*3.8e33
    prea = np.zeros(len(lumsp))
    posta = np.zeros(len(lumsp))

    for i, pre, m in zip(np.transpose(qlf.intvals), qlf.pre, qlf.slopes):
        dens = i
        if pre == True:
            prea += dens
        else:
            posta += dens

    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF')
    
    pars = 6
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1

    lumsshen = lums
    xshen = 10**lumsshen*3.8e33

    dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'a')
    
    ax.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
    
#     ax.axvline(10**8.95*3.8e33,c='k',linestyle='dotted')
    ax.axvline(10**6.5*3.8e33,c='r',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
    ### formatting and save
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-3,'z = '+str(z),fontsize = 12)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    
f = h5py.File("output/chi2_SHEN_r10_v2.2.2_w0.h5py", "r") 
logMstar0 = f['logMstar0'][:]
siglnX2 = f['siglnX2'][:]
siglnX1 = f['siglnX1'][:]
slope_low = f['slope_low'][:]
norm_from_local = f['norm_from_local'][:]
chi23d = f['z=1.0/chi23d_grid'][:].T
f.close()
minval = np.amin(chi23d)
minind = np.where(chi23d == minval)
bestnorm = norm_from_local[minind[0][0]]
bestslope = slope_low[minind[1][0]]
bestpost = siglnX2[minind[2][0]]
bestpre = siglnX1[minind[3][0]]
bestcrit = logMstar0[minind[4][0]]

print('\nShen best fits (orig, stacked z): minval =',minval)
print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f}\n ')


fig, (ax, ax2, ax3, ax4) = plt.subplots(1,4,figsize=(15.5,3.5))

fit_params = [0.2, 0.005, bestcrit, [bestpre, bestpost], bestslope, bestnorm, np.linspace(5,18,200)]

# ax.text(10**48.75, -4, '$\sigma_{\ln X} = $'+str(fit_params[3][1])[0:5])
QLFwShen_test(z=1, fit_params=fit_params, approx_local = True)

pars = 6
cs = list(cm.Greens(np.arange(pars) / pars) ) 


qlf = QLF(1, fit_params[1])
qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
qlf.get_dNdlnL(fit_params[-1], fit_params[3])
ax2.plot(10**qlf.StellBins, qlf.SSFRs,c='k')
ax2.set_ylabel('Observed SSFR (yr$^{-1}$)',fontsize=12)
ax2.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim([10**7,10**12.5])
ax2.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax2.axvline(10**mass, ls = 'dotted', c=c)
    ax2.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label=f'{mass}'+r'$< \log [M_*](M_\odot) <$'+f'{mass+1}')
ax2.legend(fontsize = 8, framealpha = 1)



ax3.plot(10**qlf.StellBins, np.log10(qlf.dNdlogMstar), c='k')
ax3.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [M_*]^{-1})$',fontsize=12)
ax3.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax3.set_xscale('log')
ax3.set_xlim([10**7,10**12.5])
ax3.set_ylim([-7,0])
ax3.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax3.axvline(10**mass, ls = 'dotted', c=c)
    ax3.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)

    
ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='~Local')
ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
ax4.set_yscale('log')
ax4.set_xscale('log')
ax4.set_xlim([10**7,10**12.5])
ax4.set_ylim([10**3, 10**10])
ax4.legend(fontsize = 8, framealpha=1)
ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax4.axvline(10**mass, ls = 'dotted', c=c)
    ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)

plt.tight_layout()

plt.savefig('plots/SMFstart_bestfit-extendedL.pdf', transparent = True)




Shen best fits (orig, stacked z): minval = 6.957098590371464
Best crit = 10.00,  Best pre-disk = 2.00, Best post-disk = 2.00,  Best slope = 0.84,  Best norm = 1.33
 


FigureCanvasNbAgg()

In [83]:
'''
SINGLE QLF
'''
from functions import *
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import h5py
from matplotlib.backends.backend_pdf import PdfPages
%matplotlib widget


def Shen_fit_uncer(z, lums, ver): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params

    def shen_func(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(z, [d0]) + C.chebval(1 + z, [0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_b(p):
        L = lums
        a0, a1, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = a0*zfrac**a1
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)

    params = {'a0':[0.85858, 0.03092, 0.02876], 'a1':[-0.26236, 0.02003, 0.01753], 'a2':[0.02105, 0.00136, 0.00113],\
        'b0':[2.54992, 0.01915, 0.02949], 'b1':[-1.04735, 0.01815, 0.02999], 'b2':[1.13277, 0.01988, 0.03891],\
        'c0':[13.01297, 0.00943, 0.01354], 'c1':[-0.57587, 0.00205, 0.00261], 'c2':[0.45361, 0.00290, 0.00434],\
        'd0':[-3.53138, 0.02694, 0.02690], 'd1':[-0.39961, 0.00871, 0.00896]}
    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    
    params_b = {'a0':[0.3653, 0.0115, 0.0114], 'a1':[-0.6006, 0.0422, 0.0417],\
        'b0':[2.4709,0.0163,0.0169], 'b1':[-0.9963,0.0167,0.0161], 'b2':[1.0716, 0.0180, 0.0181],\
        'c0':[12.9656,0.0092,0.0089], 'c1':[-0.5758,0.0020,0.0019], 'c2':[0.4698,0.0025,0.0026],\
        'd0':[-3.6276,0.0209, 0.0203], 'd1':[-0.3444,0.0063,0.0061]}
    
    
    if ver == 'orig':
        param_list = np.array([params[i] for i in params])
        NUM = int(1e4)
        rand_params = get_params(params)
        ys = np.apply_along_axis(shen_func, 1, rand_params).T
        ya = shen_func(param_list[:,0])
    elif ver == 'a':
        param_list = np.array([params_a[i] for i in params_a])
        NUM = int(1e4)
        rand_params = get_params(params_a)
        ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
        ya = shen_func_a(param_list[:,0]) 
    elif ver == 'b':
        param_list = np.array([params_b[i] for i in params_b])
        NUM = int(1e4)
        rand_params = get_params(params_b)
        ys = np.apply_along_axis(shen_func_b, 1, rand_params).T
        ya = shen_func_b(param_list[:,0]) 

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw





def QLFwShen_test(fit_params = None, z = 0.0, name = 'z0-QLF-v-Shen.pdf', Hopkins = False, approx_local=True):
   

    ### what fit params are we using
    if not fit_params:
        siglnM = 0.7
        bins = 0.005
        start = 10.0
        siglnX = [3.0, 2.0]
        lums = np.linspace(5,18,200)
    else:
        siglnM, bins, start, siglnX, slope, norm, lums = fit_params
    
    qlf = QLF(z, bins)
    qlf.get_Mbh(start, slope, norm, approx_local=approx_local)

    m = qlf.slopes

    qlf.get_dNdlnL(lums, siglnX)
    lumsp = 10**lums*3.8e33
    prea = np.zeros(len(lumsp))
    posta = np.zeros(len(lumsp))

    for i, pre, m in zip(np.transpose(qlf.intvals), qlf.pre, qlf.slopes):
        dens = i
        if pre == True:
            prea += dens
        else:
            posta += dens

    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF')
    
    pars = 6
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1

    lumsshen = lums
    xshen = 10**lumsshen*3.8e33

    dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'a')
    
    ax.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
    
#     ax.axvline(10**8.95*3.8e33,c='k',linestyle='dotted')
    ax.axvline(10**6.5*3.8e33,c='r',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
    ### formatting and save
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'QLF ($\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$)', fontsize =12)
    ax.text(10**48.75,-3,'z = '+str(z),fontsize = 12)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    
f = h5py.File("output/chi2_SHEN_r10_v2.2.2_w0.h5py", "r") 
logMstar0 = f['logMstar0'][:]
siglnX2 = f['siglnX2'][:]
siglnX1 = f['siglnX1'][:]
slope_low = f['slope_low'][:]
norm_from_local = f['norm_from_local'][:]
chi23d = f['z=1.0/chi23d_grid'][:].T
f.close()
minval = np.amin(chi23d)
minind = np.where(chi23d == minval)
bestnorm = norm_from_local[minind[0][0]]
bestslope = slope_low[minind[1][0]]
bestpost = siglnX2[minind[2][0]]
bestpre = siglnX1[minind[3][0]]
bestcrit = logMstar0[minind[4][0]]

print('\nShen best fits (orig, stacked z): minval =',minval)
print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f}\n ')

fig, ([ax, ax2, ax3], [ax4, ax6, ax5]) = plt.subplots(2,3,figsize=(12.5,8))

fit_params = [0.2, 0.005, bestcrit, [bestpre, bestpost], bestslope, bestnorm, np.linspace(5,18,200)]

# ax.text(10**48.75, -4, '$\sigma_{\ln X} = $'+str(fit_params[3][1])[0:5])
QLFwShen_test(z=1, fit_params=fit_params, approx_local = True)

pars = 6
cs = list(cm.Greens(np.arange(pars) / pars) ) 
pars2 = 6
cs2 = list(cm.Blues(np.arange(pars2) / pars2) ) 


qlf = QLF(1, fit_params[1])
qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
qlf.get_dNdlnL(fit_params[-1], fit_params[3])

CRIT_IND = np.argmin(np.abs(qlf.StellBins - fit_params[2]))

ax2.plot(10**qlf.StellBins, qlf.SSFRs,c='k')
ax2.set_ylabel('Observed SSFR (yr$^{-1}$)',fontsize=12)
ax2.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim([10**7,10**12.5])
ax2.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax2.axvline(10**mass, ls = 'dotted', c=c)
    ax2.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label=f'{mass}'+r'$< \log [M_*](M_\odot) <$'+f'{mass+1}')
ax2.legend(fontsize = 8, framealpha = 1)
ax2.axvline(10**qlf.StellBins[CRIT_IND], color='y')



ax3.plot(10**qlf.StellBins, np.log10(qlf.dNdlogMstar), c='k')
ax3.set_ylabel(r'SMF: $\log_{10} \Phi (Mpc^{-3} \log_{10} [M_*]^{-1})$',fontsize=12)
ax3.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax3.set_xscale('log')
ax3.set_xlim([10**7,10**12.5])
ax3.set_ylim([-7,0])
ax3.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax3.axvline(10**mass, ls = 'dotted', c=c)
    ax3.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
ax3.axvline(10**qlf.StellBins[CRIT_IND], color='y')

    
ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='~Local')
ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
ax4.set_yscale('log')
ax4.set_xscale('log')
ax4.set_xlim([10**7,10**12.5])
ax4.set_ylim([10**4, 10**10])
ax4.legend(fontsize = 8, framealpha=1)
ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax4.axvline(10**mass, ls = 'dotted', c=c)
    ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
for c, mass in zip(cs2, [4,5,6,7,8,9]):
    ax4.axhline(10**mass, ls = 'dotted', c=c)
    ax4.axhspan(10**mass, 10**(mass+1), color=c, alpha = 0.25)
ax4.axvline(10**qlf.StellBins[CRIT_IND], color='y')
ax4.axhline(10**qlf.BHBins[CRIT_IND], color='y')
    
    
BHMF = qlf.dNdlogMstar / qlf.slopes
ax5.plot(10**qlf.BHBins, np.log10(BHMF), c='k')
ax5.set_ylabel(r'BHMF: $\log_{10} \Phi (Mpc^{-3} \log_{10} [M_{BH}]^{-1})$',fontsize=12)
ax5.set_xlabel('$M_{BH}(M_\odot)$',fontsize=12)
ax5.set_xscale('log')
ax5.set_xlim([10**4, 10**10])
ax5.set_ylim([-7,0])
ax5.text(10**6, -1.0, r"$\frac{dN}{d\logM_{BH}} = \frac{dN}{d\logM_*} \frac{d\lnM_{*}}{d\lnM_{BH}}$", fontsize = 12)
for c, mass in zip(cs2, [4,5,6,7,8,9]):
    ax5.axvline(10**mass, ls = 'dotted', c=c)
    ax5.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label=f'{mass}'+r'$< \log [M_{BH}](M_\odot) <$'+f'{mass+1}')
ax5.legend(fontsize = 8, framealpha = 1)
ax5.axvline(10**qlf.BHBins[CRIT_IND], color='y')



ax6.plot(10**qlf.BHBins, qlf.slopes, c='k')
ax6.set_xlabel('$M_{BH}(M_\odot)$',fontsize=12)
ax6.set_ylabel(r'$\frac{d\lnM_{BH}}{d\lnM_{*}}$')
ax6.set_xscale('log')
ax6.set_xlim([10**4, 10**10])
for c, mass in zip(cs2, [4,5,6,7,8,9]):
    ax6.axvline(10**mass, ls = 'dotted', c=c)
    ax6.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
ax6.axvline(10**qlf.BHBins[CRIT_IND], color='y')

plt.tight_layout()

plt.savefig('plots/SMFstart_bestfit_wBHMF.pdf', transparent = True)




Shen best fits (orig, stacked z): minval = 6.957098590371464
Best crit = 10.00,  Best pre-disk = 2.00, Best post-disk = 2.00,  Best slope = 0.84,  Best norm = 1.33
 


FigureCanvasNbAgg()

In [21]:
'''
SINGLE QLF
'''
from functions_newparams import *
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import h5py
from matplotlib.backends.backend_pdf import PdfPages
%matplotlib widget


def Shen_fit_uncer(z, lums, ver): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params

    def shen_func(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(z, [d0]) + C.chebval(1 + z, [0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    def shen_func_b(p):
        L = lums
        a0, a1, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = a0*zfrac**a1
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)

    params = {'a0':[0.85858, 0.03092, 0.02876], 'a1':[-0.26236, 0.02003, 0.01753], 'a2':[0.02105, 0.00136, 0.00113],\
        'b0':[2.54992, 0.01915, 0.02949], 'b1':[-1.04735, 0.01815, 0.02999], 'b2':[1.13277, 0.01988, 0.03891],\
        'c0':[13.01297, 0.00943, 0.01354], 'c1':[-0.57587, 0.00205, 0.00261], 'c2':[0.45361, 0.00290, 0.00434],\
        'd0':[-3.53138, 0.02694, 0.02690], 'd1':[-0.39961, 0.00871, 0.00896]}
    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    
    params_b = {'a0':[0.3653, 0.0115, 0.0114], 'a1':[-0.6006, 0.0422, 0.0417],\
        'b0':[2.4709,0.0163,0.0169], 'b1':[-0.9963,0.0167,0.0161], 'b2':[1.0716, 0.0180, 0.0181],\
        'c0':[12.9656,0.0092,0.0089], 'c1':[-0.5758,0.0020,0.0019], 'c2':[0.4698,0.0025,0.0026],\
        'd0':[-3.6276,0.0209, 0.0203], 'd1':[-0.3444,0.0063,0.0061]}
    
    
    if ver == 'orig':
        param_list = np.array([params[i] for i in params])
        NUM = int(1e4)
        rand_params = get_params(params)
        ys = np.apply_along_axis(shen_func, 1, rand_params).T
        ya = shen_func(param_list[:,0])
    elif ver == 'a':
        param_list = np.array([params_a[i] for i in params_a])
        NUM = int(1e4)
        rand_params = get_params(params_a)
        ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
        ya = shen_func_a(param_list[:,0]) 
    elif ver == 'b':
        param_list = np.array([params_b[i] for i in params_b])
        NUM = int(1e4)
        rand_params = get_params(params_b)
        ys = np.apply_along_axis(shen_func_b, 1, rand_params).T
        ya = shen_func_b(param_list[:,0]) 

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw





def QLFwShen_test(fit_params = None, z = 0.0, name = 'z0-QLF-v-Shen.pdf', Hopkins = False, approx_local=True):
   

    ### what fit params are we using
    if not fit_params:
        siglnM = 0.7
        bins = 0.005
        start = 10.0
        siglnX = [3.0, 2.0]
        lums = np.linspace(5,18,200)
    else:
        siglnM, bins, start, siglnX, slope, norm, lums = fit_params
    
    qlf = QLF(z, bins)
#     qlf.SSFRs[:] = qlf.SSFRs[0]
    qlf.get_dNdlnMstar(fit_params[0])
    qlf.get_Mbh(start, slope, norm, approx_local=approx_local)

    m = qlf.slopes

    qlf.get_dNdlnL(lums, siglnX)
    lumsp = 10**lums*3.8e33
    prea = np.zeros(len(lumsp))
    posta = np.zeros(len(lumsp))

    for i, pre, m in zip(np.transpose(qlf.intvals), qlf.pre, qlf.slopes):
        dens = i
        if pre == True:
            prea += dens
        else:
            posta += dens

#     l1, = ax.plot(lumsp, np.log10(prea*np.log(10)), lw=1.2, c='r', linestyle='dashed', label='Pre-Disk')
#     l2, = ax.plot(lumsp, np.log10(posta*np.log(10)), lw=1.2, c='b', linestyle='dashed', label='Post-Disk')
    ax.plot(lumsp, np.log10(qlf.dNdlnL * np.log(10)), c='k', label = 'Predicted QLF',linestyle='solid')
    
    
    ####### HERE IS THE SHIFTED QLF PREDICTED LINE
    #######
    ## shifting just y
#     ax.plot(lumsp, np.log10(qlf.dNdlnL * np.log(10))-0.6, c='gray', label = 'Predicted QLF (shifted)',linestyle='dotted',lw = 2.5)

    ## shifting both x and y
#     ax.plot(10**(np.log10(lumsp)+0.4), np.log10(qlf.dNdlnL * np.log(10))-0.8, c='gray', label = 'Predicted QLF (shifted)',linestyle='dotted',lw = 2.5)

    #######
    #######
    
    pars = 6
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp*np.log(10)), lw = 1.2, color = c, linestyle='dotted')#, label=f'{mass_begin}'+r'$< \log [M_*](M_\odot) <$'+f'{mass_begin+1}')
        mass_begin += 1

    ### plotting Hopkins data (if told to)
    if Hopkins == True:
        x,y,yerr = grab_obs(z)
        ax.errorbar(10**np.asarray(x)*3.8e33,y,yerr=yerr,markersize=1,fmt='o',c='gray',label='Hopkins+2006')
        
    lumsshen = np.linspace(8.95,14.95,200) ## this is tenative and an approximate range of valid observational data
    lumsshen = lums
    xshen = 10**lumsshen*3.8e33

    ### plotting Shen data and our QLF
#     dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'orig')
#     ax.plot(xshen, dens, label='Shen Orig',c='k',lw=2,alpha = 0.5)
#     ax.fill_between(xshen, dens-stanab, dens+stanb, color='k', alpha=.2)
    dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'a')
    
    #############
    i = np.argmin(lumsp-10**42)
    norm = np.log10(qlf.dNdlnL[i] * np.log(10)) - dens[i]
#     ax.plot(lumsp, np.log10(qlf.dNdlnL * np.log(10))-norm, c='gray', label = 'Predicted QLF (shifted)',linestyle='dotted',lw = 2.5)
    #############
    
    ax.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
#     dens, stanave, stanab, stanb = Shen_fit_uncer(z, lumsshen, 'b')
#     ax.plot(xshen, dens, label='Shen Global B',c='brown',lw=2,alpha = 0.5)
#     ax.fill_between(xshen, dens-stanab, dens+stanb, color='brown', alpha=.2)
    
    ax.axvline(10**8.95*3.8e33,c='k',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
    ### formatting and save
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**7.5*3.8e33,-1.5,'z = '+str(z),fontsize = 12)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    
#     
f = h5py.File("output/chi2_3pShenfit_15_newparams-mk2.h5py", "r") 
# f = h5py.File("output/chi2_3pShenfit_15_constantSSFRtest.h5py", "r")
logMstar0 = f['logMstar0'][:]
siglnX2 = f['siglnX2'][:]
siglnX1 = f['siglnX1'][:]
slope_low = f['slope_low'][:]
norm_from_local = f['norm_from_local'][:]
chi23d = f['z=1.0/chi23d_grid'][:].T
f.close()
minval = np.amin(chi23d)
minind = np.where(chi23d == minval)
bestnorm = norm_from_local[minind[0][0]]
bestslope = slope_low[minind[1][0]]
bestpost = siglnX2[minind[2][0]]
bestpre = siglnX1[minind[3][0]]
bestcrit = logMstar0[minind[4][0]]

print('\nShen best fits (orig, stacked z): minval =',minval)
print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f}\n ')


fig, (ax,ax2,ax3) = plt.subplots(1,3,figsize=(18,4))

fit_params = [0.0, 0.005, 3, [1.8, 1.8], 0.90, 1.0, np.linspace(5,18,200)]
ax.text(10**49, -4, '$\sigma_{\ln X} = $'+str(fit_params[3][1])[0:5])
# fit_params = [0.2, 0.005, bestcrit, [bestpre, bestpost], bestslope, bestnorm, np.linspace(5,18,200)]

QLFwShen_test(z=1, fit_params=fit_params, approx_local = True)

pars = 6
cs = list(cm.Greens(np.arange(pars) / pars) ) 


qlf = QLF(1, fit_params[1])
# qlf.SSFRs[:] = qlf.SSFRs[0]
qlf.get_dNdlnMstar(fit_params[0])

qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
qlf.get_dNdlnL(fit_params[-1], fit_params[3])
ax2.plot(10**qlf.StellBins, qlf.SSFRs,c='k')

ax2.set_ylabel('Observed SSFR (yr$^{-1}$)',fontsize=12)
ax2.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim([10**7,10**14])
ax2.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax2.axvline(10**mass, ls = 'dotted', c=c)
    ax2.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5, label=f'{mass}'+r'$< \log [M_*](M_\odot) <$'+f'{mass+1}')
    ax2.legend(fontsize = 8, framealpha = 1)



ax3.plot(10**qlf.StellBins, np.log10(qlf.dNdlnMstar), c='k')
ax3.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [M_*]^{-1})$',fontsize=12)
ax3.set_xlabel('$M_*(M_\odot)$',fontsize=12)
ax3.set_xscale('log')
ax3.set_xlim([10**7,10**14])
ax3.set_ylim([-7,0])
ax3.tick_params(axis='both', which='both', labelsize=10, direction='in')
for c, mass in zip(cs, [7,8,9,10,11,12,13]):
    ax3.axvline(10**mass, ls = 'dotted', c=c)
    ax3.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)



#     plt.savefig('QLF_test_constant_ssfr_loglinearMbhMstar_shift-y.pdf')
# plt.savefig('plots/HMFstart.pdf',transparent=True)



Shen best fits (orig, stacked z): minval = 7.175656980500515
Best crit = 10.36,  Best pre-disk = 1.00, Best post-disk = 2.29,  Best slope = 0.86,  Best norm = 1.50
 


FigureCanvasNbAgg()

#### I am calculating the following:

#### $\frac{\rm{dN}}{\rm{d}\ln \rm{L}_{bol}} = \int p[\ln \rm{L}_{bol} | \ln \rm{M}_* ] \frac{dN}{d\ln \rm{M}_*} d\ln \rm{M}_*$

#### Universe Machine observational smf data I interpolate gives me...

#### $\frac{\rm{dN}}{\rm{d}\ln \rm{M}_*}$

#### ...? 

### *** No, the UM data is log10 not ln. And we can calculate ln by:

### $\frac{\rm{dN}}{\rm{d}\log_{10} \rm{M}_*} = \frac{\rm{dN}}{\rm{d}\ln \rm{M}_*} \times \ln 10$

### and

### $\ln 10 = \frac{1}{\log_{10}e}$ ***

#### If so then I just plug that in. But these values are associated with $\log_{10} \rm{M}_*$.

#### And 

#### $\rm{d}\log_{10} \rm{M}_* \neq \rm{d}\ln \rm{M}_*$ 

#### so I need to use 

#### $\rm{d}\ln \rm{M}_* = \rm{d}\log_{10} \rm{M}_* \times \ln 10$? 

In [29]:
def Shen_fit_uncer(z, lums): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)
        return np.log10(Phibol)

    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    
    param_list = np.array([params_a[i] for i in params_a])
    NUM = int(1e4)
    rand_params = get_params(params_a)
    ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
    ya = shen_func_a(param_list[:,0]) 

    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw


%matplotlib widget

        
lumsshen = np.linspace(8.95,14.95,200) ## this is tenative and an approximate range of valid observational data
xshen = 10**lumsshen*3.8e33

dens, stanave, stanab, stanb = Shen_fit_uncer(1.0, lumsshen)


plt.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
plt.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
plt.axis([1e40, 1e51, -10.5, -3.5])
plt.xscale('log')



FigureCanvasNbAgg()

In [5]:
def get_QLF():
    qlf = QLF(1, fit_params[1])
    qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
    qlf.get_dNdlnL(fit_params[-1], fit_params[3])

In [6]:
%prun get_QLF()

         12760 function calls (12758 primitive calls) in 0.051 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      200    0.013    0.000    0.013    0.000 functions_SMFstart.py:214(gauss_Mdot)
        2    0.009    0.005    0.044    0.022 shape_base.py:270(apply_along_axis)
     1900    0.008    0.000    0.008    0.000 functions_SMFstart.py:203(get_Mdotbh)
     2117    0.006    0.000    0.006    0.000 {built-in method numpy.array}
        1    0.003    0.003    0.047    0.047 functions_SMFstart.py:224(get_dNdlnL)
     2102    0.003    0.000    0.003    0.000 index_tricks.py:653(__next__)
        1    0.002    0.002    0.002    0.002 functions_SMFstart.py:105(__init__)
     2102    0.002    0.000    0.005    0.000 shape_base.py:373(<genexpr>)
2104/2102    0.001    0.000    0.001    0.000 {built-in method builtins.next}
        7    0.001    0.000    0.001    0.000 {built-in method numpy.zeros}
     2106    0.001    0.000  

In [7]:
from functions_newparams import *
def get_QLF():
    qlf = QLF(1, fit_params[1])
    qlf.get_dNdlnMstar(fit_params[0])
    qlf.get_Mbh(fit_params[2], fit_params[4], fit_params[5], approx_local=True)
    qlf.get_dNdlnL(fit_params[-1], fit_params[3])
    
%prun get_QLF()

         37751 function calls (36747 primitive calls) in 0.592 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1901    0.139    0.000    0.139    0.000 functions_newparams.py:166(get_Mstar)
        3    0.089    0.030    0.089    0.030 {built-in method scipy.interpolate._fitpack._spl_}
        2    0.079    0.039    0.079    0.039 peaks.py:70(lagrangianR)
        1    0.067    0.067    0.067    0.067 mass_function.py:958(modelDespali16)
        1    0.051    0.051    0.556    0.556 functions_newparams.py:189(convolve_smhm)
     1900    0.045    0.000    0.045    0.000 functions_newparams.py:182(gauss_array)
        2    0.039    0.019    0.132    0.066 cosmology.py:2140(sigma)
        4    0.019    0.005    0.241    0.060 shape_base.py:270(apply_along_axis)
      200    0.011    0.000    0.011    0.000 functions_newparams.py:282(gauss_Mdot)
     5938    0.006    0.000    0.006    0.000 {built-in method numpy.array}
   